# TP06: Feature Engineering y Modelado
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 3: Herramientas en la Nube de Modelado de Datos

---

### 🎯 Objetivos del Trabajo Práctico

1. Realizar **ingeniería de características** (feature engineering)
2. **Estandarizar variables numéricas**
3. Entrenar un **modelo predictivo** con Scikit-learn
4. Evaluar el **desempeño del modelo**
5. Interpretar **resultados y predicciones**

---

### 📁 Caso de Estudio: Predicción de Ventas

Desarrollaremos un modelo para predecir las ventas futuras de la panadería.

### 🕰️ Duración Estimada: 3 horas

In [0]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Librerías importadas")
print("Listo para feature engineering y modelado")

## Parte 1: Preparar Dataset para Modelado

### 📂 Consolidar datos para predicción

Vamos a crear un dataset consolidado que incluya features relevantes para predecir la facturación diaria.

In [0]:
# Cargar datos
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_ventas = pd.read_csv(ruta_datos + 'ventas.csv', parse_dates=['fecha'])
df_detalles = pd.read_csv(ruta_datos + 'detalles_ventas.csv')
df_productos = pd.read_csv(ruta_datos + 'productos.csv')
df_sucursales = pd.read_csv(ruta_datos + 'sucursales.csv')

# Unir para crear dataset completo
df_completo = df_detalles.merge(df_ventas[['venta_id', 'fecha', 'sucursal_id']], on='venta_id')
df_completo = df_completo.merge(df_productos[['producto_id', 'categoria']], on='producto_id')
df_completo = df_completo.merge(df_sucursales[['sucursal_id', 'zona']], on='sucursal_id')

print(f"✅ Dataset consolidado: {len(df_completo):,} registros")
print(f"\n📌 Columnas disponibles: {list(df_completo.columns)}")
df_completo.head()

In [0]:
# Agregar ventas por día para crear el dataset de entrenamiento
# Objetivo: predecir la facturación total de un día

df_diario = df_completo.groupby(['fecha', 'sucursal_id', 'zona']).agg({
    'venta_id': 'nunique',  # número de transacciones
    'subtotal': 'sum'  # facturación total
}).rename(columns={
    'venta_id': 'numero_ventas',
    'subtotal': 'facturacion_total'
}).reset_index()

print(f"✅ Dataset diario creado: {len(df_diario):,} registros")
print(f"\n📊 Primeras filas:")
display(df_diario.head(10))

## Parte 2: Ingeniería de Características (Feature Engineering)

### 🏭 Crear features predictivas

Vamos a crear características que ayuden a predecir la facturación: día de semana, mes, tendencias, etc.

In [0]:
# Crear features temporales
df_diario['fecha'] = pd.to_datetime(df_diario['fecha'])

# Features de tiempo
df_diario['anio'] = df_diario['fecha'].dt.year
df_diario['mes'] = df_diario['fecha'].dt.month
df_diario['dia'] = df_diario['fecha'].dt.day
df_diario['dia_semana'] = df_diario['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo
df_diario['es_fin_semana'] = (df_diario['dia_semana'] >= 5).astype(int)
df_diario['trimestre'] = df_diario['fecha'].dt.quarter
df_diario['semana_del_anio'] = df_diario['fecha'].dt.isocalendar().week

# Feature de días desde inicio
df_diario['dias_desde_inicio'] = (df_diario['fecha'] - df_diario['fecha'].min()).dt.days

print("✅ Features temporales creadas")
print(f"\n📊 Nuevas columnas:")
print([col for col in df_diario.columns if col not in ['fecha', 'sucursal_id', 'zona', 'numero_ventas', 'facturacion_total']])

In [0]:
# Crear features de tendencia (promedios móviles)
df_diario = df_diario.sort_values(['sucursal_id', 'fecha'])

# Promedio móvil de 7 días (semana anterior)
df_diario['facturacion_ma7'] = df_diario.groupby('sucursal_id')['facturacion_total'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# Promedio móvil de 30 días (mes anterior)
df_diario['facturacion_ma30'] = df_diario.groupby('sucursal_id')['facturacion_total'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean()
)

# Diferencia con día anterior (lag)
df_diario['facturacion_lag1'] = df_diario.groupby('sucursal_id')['facturacion_total'].shift(1)
df_diario['facturacion_diff'] = df_diario['facturacion_total'] - df_diario['facturacion_lag1']

print("✅ Features de tendencia creadas")
print("\n📊 Muestra con nuevas features:")
display(df_diario[['fecha', 'sucursal_id', 'facturacion_total', 'facturacion_ma7', 'facturacion_ma30']].head(10))

## Parte 3: Encoding y Escalado de Variables

### 🔢 Transformar variables para ML

Los modelos de machine learning requieren que todas las variables sean numéricas y estén en escalas similares.

In [0]:
# Encoding de variables categóricas con LabelEncoder
le_zona = LabelEncoder()
df_diario['zona_encoded'] = le_zona.fit_transform(df_diario['zona'])

le_sucursal = LabelEncoder()
df_diario['sucursal_encoded'] = le_sucursal.fit_transform(df_diario['sucursal_id'])

print("✅ Variables categóricas codificadas")
print("\n📊 Mapeos:")
print("Zonas:", dict(zip(le_zona.classes_, le_zona.transform(le_zona.classes_))))
print("Sucursales:", dict(zip(le_sucursal.classes_, le_sucursal.transform(le_sucursal.classes_))))

In [0]:
# Eliminar filas con valores nulos (primeros días sin lags)
df_modelo = df_diario.dropna().copy()

# Seleccionar features para el modelo
features = [
    'sucursal_encoded', 'zona_encoded',
    'anio', 'mes', 'dia', 'dia_semana', 'es_fin_semana', 'trimestre', 'semana_del_anio',
    'dias_desde_inicio', 'numero_ventas',
    'facturacion_ma7', 'facturacion_ma30', 'facturacion_lag1'
]

target = 'facturacion_total'

X = df_modelo[features]
y = df_modelo[target]

print(f"✅ Dataset preparado para modelado")
print(f"\n📊 Dimensiones:")
print(f"  Features (X): {X.shape}")
print(f"  Target (y): {y.shape}")
print(f"\n📌 Features seleccionadas:")
print(features)

In [0]:
# Dividir en train y test (80% - 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False  # shuffle=False para respetar orden temporal
)

print(f"✅ Dataset dividido")
print(f"  Train: {X_train.shape[0]:,} registros")
print(f"  Test: {X_test.shape[0]:,} registros")

# Escalar features numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Features escaladas con StandardScaler")

## Parte 4: Entrenamiento del Modelo Predictivo

### 🤖 Random Forest Regressor

Usaremos Random Forest, un algoritmo robusto que maneja bien relaciones no lineales y variables categóricas.

In [0]:
# Entrenar modelo Random Forest
print("🤖 Entrenando modelo Random Forest...")

model = RandomForestRegressor(
    n_estimators=100,  # número de árboles
    max_depth=10,      # profundidad máxima
    min_samples_split=5,
    random_state=42,
    n_jobs=-1  # usar todos los cores
)

model.fit(X_train_scaled, y_train)

print("✅ Modelo entrenado exitosamente")
print(f"\n🌳 Número de árboles: {model.n_estimators}")
print(f"🌳 Profundidad máxima: {model.max_depth}")

In [0]:
# Realizar predicciones
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

print("✅ Predicciones realizadas")
print(f"\n📄 Comparación (primeras 10 predicciones del test):")

comparacion = pd.DataFrame({
    'Real': y_test.values[:10],
    'Predicho': y_test_pred[:10],
    'Diferencia': y_test.values[:10] - y_test_pred[:10],
    'Error_%': np.abs((y_test.values[:10] - y_test_pred[:10]) / y_test.values[:10] * 100).round(2)
})

display(comparacion)

In [0]:
# Calcular métricas de evaluación
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print("📊 MÉTRICAS DE EVALUACIÓN")
print("=" * 80)
print(f"\nTrain Set:")
print(f"  MAE (Mean Absolute Error):  ${mae_train:,.2f}")
print(f"  RMSE (Root Mean Squared Error): ${rmse_train:,.2f}")
print(f"  R² Score: {r2_train:.4f}")

print(f"\nTest Set:")
print(f"  MAE (Mean Absolute Error):  ${mae_test:,.2f}")
print(f"  RMSE (Root Mean Squared Error): ${rmse_test:,.2f}")
print(f"  R² Score: {r2_test:.4f}")

print("\n" + "=" * 80)
print("\n💡 Interpretación:")
print(f"  - El modelo predice con un error promedio de ${mae_test:,.2f}")
print(f"  - Explica el {r2_test*100:.1f}% de la varianza en las ventas")

In [0]:
# Visualizar predicciones vs valores reales
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Scatter plot: Real vs Predicho
ax1.scatter(y_test, y_test_pred, alpha=0.5, s=20)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
ax1.set_xlabel('Facturación Real ($)', fontsize=12)
ax1.set_ylabel('Facturación Predicha ($)', fontsize=12)
ax1.set_title('🎯 Predicciones vs Valores Reales', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Distribución de errores
errores = y_test - y_test_pred
ax2.hist(errores, bins=50, edgecolor='black', alpha=0.7, color='coral')
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Error = 0')
ax2.set_xlabel('Error de Predicción ($)', fontsize=12)
ax2.set_ylabel('Frecuencia', fontsize=12)
ax2.set_title('📉 Distribución de Errores', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Analizar importancia de features
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("🎯 IMPORTANCIA DE FEATURES")
print("=" * 80)
display(feature_importance)

# Visualizar top 10 features más importantes
fig, ax = plt.subplots(figsize=(10, 6))
top_features = feature_importance.head(10)
ax.barh(range(len(top_features)), top_features['importance'], color='steelblue', edgecolor='black')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.set_xlabel('Importancia', fontsize=12)
ax.set_title('🏆 Top 10 Features Más Importantes', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 🎯 Resumen del TP06

### ✅ Qué aprendimos:

1. **Feature Engineering**: Creamos features temporales, de tendencia y lags
2. **Encoding**: Transformamos variables categóricas con LabelEncoder
3. **Escalado**: Normalizamos features con StandardScaler
4. **Modelado**: Entrenamos un Random Forest Regressor
5. **Evaluación**: Calculamos MAE, RMSE y R² Score
6. **Interpretación**: Analizamos importancia de features y errores

### 📊 Resultados del modelo:

* MAE: Error promedio en pesos
* RMSE: Penaliza errores grandes
* R²: Porcentaje de varianza explicada
* Features más importantes: promedios móviles, lags, día de semana

### 🚀 Próximos pasos:

En la **Unidad 4** (Proyectos Integradores) aprenderemos a:
* Crear pipelines completos de datos
* Automatizar procesos de ETL
* Orquestar workflows complejos
* Desarrollar proyectos end-to-end

---

**📝 Excelente! Has completado la Unidad 3 - Modelado de Datos.**

### 📚 UNIDADES 1, 2 Y 3 COMPLETADAS ✅

* **Unidad 1 - Análisis de Datos**: Carga, transformación y exploración
* **Unidad 2 - Visualización de Datos**: Perfilado y dashboards
* **Unidad 3 - Modelado de Datos**: Estructuración y feature engineering

Solo falta la **Unidad 4 - Proyectos Integradores** para completar el curso!